# 3 · Train, evaluate and export the models

Uses the search result from notebook 2 (`data/models/hyperparameters.json`, or defaults if it does not exist yet) and

1. evaluates each model **leave-one-sausage-out**: train on four sausages, score on the fifth;
2. saves one model per fold (`data/models/folds/`). The demonstrator replays a sausage with the model that never saw it;
3. trains the final models on all annotated sausages (`data/models/final/`);
4. writes `manifest.json` and `metrics.json`, which the demonstrator's evaluation panel shows.

`LIMIT_SIZE` caps the Random Forest (100 trees, depth 10). An unlimited forest on this data grows to hundreds of MB.
**Train with the same scikit-learn version the demo laptop runs**, because model files are not portable across versions.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import json
import pandas as pd
from ml_core import data
from ml_core.training import load_windows, metrics_table, train_and_evaluate

In [2]:
CONVERTED_DIR = data.CONVERTED_DIR
MODELS_DIR = data.MODELS_DIR
WINDOW_SIZE, STRIDE = 10, 2
LIMIT_SIZE = True
FOLD_MODELS = True
QUICK = False

In [3]:
windows = load_windows(CONVERTED_DIR, window_size=WINDOW_SIZE, stride=STRIDE)
windows.summary()

series_1_sausage_1: 41,462 rows → 20,727 windows
series_1_sausage_2: 41,479 rows → 20,735 windows
series_1_sausage_3: 41,468 rows → 20,730 windows
series_1_sausage_4: 41,473 rows → 20,732 windows
series_1_sausage_5: 41,465 rows → 20,728 windows


,recording,rows,windows,Umroetung,Reifung Phase 1,Reifung Phase 2
0,series_1_sausage_1,41462,20727,7304,8116,5307
1,series_1_sausage_2,41479,20735,7309,8120,5306
2,series_1_sausage_3,41468,20730,7307,8118,5305
3,series_1_sausage_4,41473,20732,7308,8120,5304
4,series_1_sausage_5,41465,20728,7307,8117,5304


In [4]:
metrics = train_and_evaluate(windows, MODELS_DIR, limit_size=LIMIT_SIZE, fold_models=FOLD_MODELS, quick=QUICK)

Hyperparameters: hyperparameters.json from 2026-11-24

=== Logistic Regression ===  {'C': 100, 'class_weight': 'balanced', 'l1_ratio': 0, 'max_iter': 1000, 'solver': 'liblinear'}
  without series_1_sausage_1: balanced acc 0.717, macro-F1 0.689 (34.8 s)
  without series_1_sausage_2: balanced acc 0.808, macro-F1 0.772 (25.4 s)
  without series_1_sausage_3: balanced acc 0.757, macro-F1 0.764 (36.1 s)
  without series_1_sausage_4: balanced acc 0.753, macro-F1 0.759 (40.7 s)
  without series_1_sausage_5: balanced acc 0.860, macro-F1 0.866 (28.6 s)
  mean balanced acc 0.779 · 3.9 ms per prediction · 0.0 MB

=== Random Forest ===  {'bootstrap': True, 'class_weight': 'balanced_subsample', 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
  without series_1_sausage_1: balanced acc 0.710, macro-F1 0.680 (1.5 s)
  without series_1_sausage_2: balanced acc 0.820, macro-F1 0.826 (1.7 s)
  without series_1_sausage_3: bala

## Results (mean ± std over the held-out sausages)

In [5]:
metrics_table(metrics)

,balanced acc,macro-F1,accuracy,ms / prediction,size (MB),fit all data (s)
model,,,,,,
Logistic Regression,0.779 ± 0.050,0.770 ± 0.056,0.780 ± 0.060,3.90,0.01,40.0
Random Forest,0.740 ± 0.082,0.738 ± 0.085,0.743 ± 0.082,9.73,4.18,2.0
MLP (neural net),0.689 ± 0.039,0.659 ± 0.036,0.678 ± 0.039,3.66,0.11,56.2


In [6]:
# balanced accuracy per held-out sausage
pd.DataFrame({m["name"]: {f["held_out"]: round(f["balanced_accuracy"], 3) for f in m["folds"]}
              for m in metrics["models"].values()})

,Logistic Regression,Random Forest,MLP (neural net)
series_1_sausage_1,0.717,0.710,0.647
series_1_sausage_2,0.808,0.820,0.743
series_1_sausage_3,0.757,0.827,0.725
series_1_sausage_4,0.753,0.603,0.652
series_1_sausage_5,0.860,0.740,0.677


In [7]:
# confusion matrices, row-normalised (rows: annotation, columns: prediction)
for m in metrics["models"].values():
    cm = pd.DataFrame(m["confusion_matrix"], index=metrics["classes"], columns=metrics["classes"])
    print(m["name"])
    display((cm.div(cm.sum(axis=1), axis=0) * 100).round(1))

Logistic Regression


,Umroetung,Reifung Phase 1,Reifung Phase 2
Umroetung,90.9,9.1,0.1
Reifung Phase 1,7.3,69.5,23.2
Reifung Phase 2,0.0,26.7,73.3


Random Forest


,Umroetung,Reifung Phase 1,Reifung Phase 2
Umroetung,90.4,9.6,0.0
Reifung Phase 1,13.4,64.6,22.0
Reifung Phase 2,0.1,32.9,67.0


MLP (neural net)


,Umroetung,Reifung Phase 1,Reifung Phase 2
Umroetung,89.0,10.9,0.1
Reifung Phase 1,18.4,46.6,35.0
Reifung Phase 2,0.0,28.8,71.1


In [8]:
# how often two models give the same label on held-out windows
pd.Series(metrics["agreement_on_held_out"]).round(3)

logreg|rf     0.793
logreg|mlp    0.801
rf|mlp        0.787
dtype: float64

**Next:** start the demonstrator (`scripts/start_demo.ps1`). It loads the models from `data/models/`.
Read the scores with care: all annotated sausages come from one curing run.